# 01 — Dataset Analysis

Explore the SpiideoSynLoc dataset: splits, image sizes, annotation stats, object size distribution.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

DATA_ROOT = Path('../data/raw/SoccerNet/SpiideoSynLoc')
ANN_DIR = DATA_ROOT / 'annotations'
ANN_FHD_DIR = DATA_ROOT / 'annotations_fullhd'

## Split statistics

In [ ]:
splits = {
    'train': 'train.json',
    'val': 'val.json',
    'test': 'test.json',
    'challenge': 'challenge_public.json',
}

for name, fname in splits.items():
    path = ANN_DIR / fname
    if not path.exists():
        print(f'{name}: NOT FOUND')
        continue
    with open(path) as f:
        data = json.load(f)
    n_imgs = len(data['images'])
    n_anns = len(data.get('annotations', []))
    img = data['images'][0]
    print(f'{name:12s}: {n_imgs:>6} images, {n_anns:>7} annotations, resolution {img["width"]}x{img["height"]}')

## Annotation size distribution

Key finding: **94% of players are "small objects"** (bbox area < 1024 px² in FullHD).  
This means higher input resolution (1280+) is critical for detection.

In [ ]:
with open(ANN_FHD_DIR / 'train.json') as f:
    train = json.load(f)

areas = [ann['area'] for ann in train['annotations']]
bboxes = [ann['bbox'] for ann in train['annotations']]
widths = [b[2] for b in bboxes]
heights = [b[3] for b in bboxes]

print(f'Total annotations: {len(areas)}')
print(f'Area: min={min(areas):.0f}, median={np.median(areas):.0f}, max={max(areas):.0f}')
print(f'Width: min={min(widths):.0f}, median={np.median(widths):.0f}, max={max(widths):.0f}')
print(f'Height: min={min(heights):.0f}, median={np.median(heights):.0f}, max={max(heights):.0f}')

# COCO size thresholds (adjusted for FullHD)
small = sum(1 for a in areas if a < 1024)   # 32x32
medium = sum(1 for a in areas if 1024 <= a < 9216)  # 96x96
large = sum(1 for a in areas if a >= 9216)
print(f'\nSmall (<32x32):  {small:>6} ({100*small/len(areas):.1f}%)')
print(f'Medium:          {medium:>6} ({100*medium/len(areas):.1f}%)')
print(f'Large (>96x96):  {large:>6} ({100*large/len(areas):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(areas, bins=100, range=(0, 5000), edgecolor='black', alpha=0.7)
axes[0].axvline(1024, color='red', linestyle='--', label='small/medium (32x32)')
axes[0].set_xlabel('Bbox area (px²)')
axes[0].set_ylabel('Count')
axes[0].set_title('Annotation area distribution')
axes[0].legend()

axes[1].hist(widths, bins=80, range=(0, 100), edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Bbox width (px)')
axes[1].set_title('Width distribution')

axes[2].hist(heights, bins=80, range=(0, 200), edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Bbox height (px)')
axes[2].set_title('Height distribution')

plt.tight_layout()
plt.savefig('../reports/figures/bbox_size_distribution.png', dpi=150)
plt.show()

## Players per image

In [ ]:
players_per_img = Counter(ann['image_id'] for ann in train['annotations'])
counts = list(players_per_img.values())

print(f'Players per image: min={min(counts)}, median={np.median(counts):.0f}, max={max(counts)}')

plt.figure(figsize=(8, 4))
plt.hist(counts, bins=range(0, max(counts)+2), edgecolor='black', alpha=0.7)
plt.xlabel('Players per image')
plt.ylabel('Number of images')
plt.title('Players per image distribution (train)')
plt.tight_layout()
plt.show()

## 4K vs FullHD annotations

Original annotations are in 4K (3840x2160), images are FullHD (1920x1080).  
**CRITICAL:** Training must use `annotations_fullhd/` (scaled ×0.5).

In [ ]:
with open(ANN_DIR / 'train.json') as f:
    train_4k = json.load(f)
with open(ANN_FHD_DIR / 'train.json') as f:
    train_fhd = json.load(f)

img_4k = train_4k['images'][0]
img_fhd = train_fhd['images'][0]
ann_4k = train_4k['annotations'][0]
ann_fhd = train_fhd['annotations'][0]

print(f'4K:   image {img_4k["width"]}x{img_4k["height"]}, bbox {ann_4k["bbox"]}')
print(f'FHD:  image {img_fhd["width"]}x{img_fhd["height"]}, bbox {ann_fhd["bbox"]}')
print(f'Scale factor: {img_4k["width"] / img_fhd["width"]:.1f}x')